# Chemical Space and Scaffold Analysis

## Scientific objective
Analyze Morgan similarity, descriptor PCA, scaffold frequencies, endpoint overlap, analogue neighborhoods, and potential activity-cliff candidates.

## Inputs
- `data/processed/endpoint_records.csv`
- `data/processed/descriptors.csv`

## Expected outputs
- `figures/descriptor_pca.png`
- `tables/scaffold_frequency.csv`
- `tables/endpoint_overlap.csv`
- `data/metadata/similarity_sample.csv`

## Dependencies
RDKit, scikit-learn, matplotlib

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
PCA and optional UMAP are exploratory projections only. Tanimoto values depend on fingerprint parameters.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Pairwise similarity sampling is not a complete map of chemical space. Activity cliffs are formally analyzed in notebook 18.

## Next notebook
[08_split_generation.ipynb](./08_split_generation.ipynb)

In [1]:
from pathlib import Path
import os
import json
import warnings
import random

# Set before NumPy, scikit-learn, RDKit, or other compiled libraries.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError(
        "Run this notebook from the repository root or notebooks directory"
    )

os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import require_paths

CONFIGS = load_configs(ROOT)
os.environ["TOX_SCREEN_PROFILE"] = "smoke"
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])

# Avoid set_global_seed() here because it imports PyTorch.
random.seed(SEED)
np.random.seed(SEED)

print(
    {
        "root": str(ROOT),
        "profile": PROFILE,
        "seed": SEED,
    }
)

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'smoke', 'seed': 20260723}


In [2]:
import os
import subprocess
import sys
import textwrap
from pathlib import Path

worker_path = ROOT / "reports" / "_descriptor_pca_worker.py"
worker_path.parent.mkdir(parents=True, exist_ok=True)

worker_code = textwrap.dedent(
    r'''
    from pathlib import Path
    import json
    import os

    # Limit native numerical-library threading.
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["OPENBLAS_NUM_THREADS"] = "1"
    os.environ["NUMEXPR_NUM_THREADS"] = "1"

    import numpy as np
    import pandas as pd

    root = Path(os.environ["TOX_PROJECT_ROOT"]).resolve()
    seed = int(os.environ["TOX_PROJECT_SEED"])

    print("1. Reading endpoint records", flush=True)

    records = pd.read_parquet(
        root / "data" / "processed" / "endpoint_records.parquet"
    )

    molecules = (
        records[
            ["molecule_id", "standardized_smiles", "scaffold"]
        ]
        .drop_duplicates("molecule_id")
        .reset_index(drop=True)
        .copy()
    )

    molecules["molecule_id"] = (
        molecules["molecule_id"].astype("string")
    )

    molecules["_row_order"] = np.arange(
        len(molecules),
        dtype=np.int64,
    )

    print(
        f"2. Unique molecules: {len(molecules):,}",
        flush=True,
    )

    desc_raw = pd.read_csv(
        root / "data" / "processed" / "descriptors.csv",
        dtype={"molecule_id": "string"},
        low_memory=False,
    )

    if desc_raw["molecule_id"].isna().any():
        missing_ids = int(
            desc_raw["molecule_id"].isna().sum()
        )
        raise ValueError(
            f"descriptors.csv contains {missing_ids} missing molecule IDs"
        )

    if desc_raw["molecule_id"].duplicated().any():
        duplicated = int(
            desc_raw["molecule_id"]
            .duplicated(keep=False)
            .sum()
        )

        raise ValueError(
            f"descriptors.csv contains "
            f"{duplicated} duplicated ID rows"
        )

    desc = molecules[
        ["molecule_id", "_row_order"]
    ].merge(
        desc_raw,
        on="molecule_id",
        how="left",
        sort=False,
        validate="one_to_one",
        indicator=True,
    )

    missing = desc["_merge"].ne("both")

    if missing.any():
        examples = (
            desc.loc[missing, "molecule_id"]
            .head()
            .tolist()
        )

        raise ValueError(
            f"Missing descriptors for "
            f"{int(missing.sum())} molecules. "
            f"Examples: {examples}"
        )

    desc = (
        desc
        .sort_values("_row_order")
        .drop(columns=["_row_order", "_merge"])
        .reset_index(drop=True)
    )

    if len(desc) != len(molecules):
        raise ValueError(
            "Descriptor alignment changed the number of molecules: "
            f"{len(molecules)} before merge and {len(desc)} afterward"
        )

    print(
        "3. Descriptor alignment passed",
        flush=True,
    )

    numeric = (
        desc
        .select_dtypes(include=[np.number])
        .replace([np.inf, -np.inf], np.nan)
    )

    # Remove descriptor columns that contain no observed values.
    numeric = numeric.loc[
        :,
        numeric.notna().any(axis=0),
    ]

    if numeric.empty:
        raise ValueError(
            "No numeric descriptor columns were found"
        )

    medians = numeric.median(axis=0)
    numeric = numeric.fillna(medians)

    x = numeric.to_numpy(
        dtype=np.float64,
        copy=True,
    )

    if not np.isfinite(x).all():
        raise ValueError(
            "Descriptor matrix still contains non-finite values"
        )

    print(
        f"4. Descriptor matrix: {x.shape}",
        flush=True,
    )

    # Equivalent to the default RobustScaler transformation:
    # subtract the median and divide by the interquartile range.
    center = np.median(
        x,
        axis=0,
    )

    q1 = np.percentile(
        x,
        25,
        axis=0,
    )

    q3 = np.percentile(
        x,
        75,
        axis=0,
    )

    scale = q3 - q1

    invalid_scale = (
        ~np.isfinite(scale)
        | np.isclose(
            scale,
            0.0,
            rtol=0.0,
            atol=1e-15,
        )
    )

    scale[invalid_scale] = 1.0

    x_scaled = (
        x - center
    ) / scale

    if not np.isfinite(x_scaled).all():
        raise ValueError(
            "Robust-scaled descriptor matrix contains non-finite values"
        )

    print(
        "5. Robust scaling passed",
        flush=True,
    )

    # Center the robust-scaled matrix for PCA.
    pca_center = x_scaled.mean(axis=0)
    x_centered = x_scaled - pca_center

    # Build the covariance matrix without BLAS-backed matrix multiplication.
    number_of_rows, number_of_features = x_centered.shape
    denominator = max(1, number_of_rows - 1)
    
    covariance = np.empty(
        (number_of_features, number_of_features),
        dtype=np.float64,
    )
    
    for i in range(number_of_features):
        for j in range(i, number_of_features):
            covariance_value = float(
                np.sum(
                    x_centered[:, i]
                    * x_centered[:, j]
                )
                / denominator
            )
    
            covariance[i, j] = covariance_value
            covariance[j, i] = covariance_value
    
    print(
        "Covariance calculation passed",
        flush=True,
    )

    # Enforce exact symmetry against floating-point asymmetry.
    covariance = 0.5 * (
        covariance + covariance.T
    )

    if not np.isfinite(covariance).all():
        raise ValueError(
            "Covariance matrix contains non-finite values"
        )


    def jacobi_eigh_symmetric(
        matrix,
        tolerance=1e-10,
        max_iterations=100000,
    ):
        """Compute eigenvalues/eigenvectors of a small symmetric matrix.

        This Jacobi rotation implementation avoids numpy.linalg.eigh and
        therefore does not invoke the failing Windows LAPACK DLL.
        """

        a = np.asarray(
            matrix,
            dtype=np.float64,
        ).copy()

        if (
            a.ndim != 2
            or a.shape[0] != a.shape[1]
        ):
            raise ValueError(
                "Jacobi decomposition requires a square matrix"
            )

        if not np.isfinite(a).all():
            raise ValueError(
                "Jacobi input matrix contains non-finite values"
            )

        # Make numerical symmetry explicit.
        a = 0.5 * (
            a + a.T
        )

        n = a.shape[0]

        eigenvectors = np.eye(
            n,
            dtype=np.float64,
        )

        for iteration in range(
            1,
            max_iterations + 1,
        ):
            largest_value = 0.0
            p = 0
            q = 1

            # Find the largest absolute off-diagonal element.
            for row in range(n - 1):
                for column in range(
                    row + 1,
                    n,
                ):
                    value = abs(
                        float(a[row, column])
                    )

                    if value > largest_value:
                        largest_value = value
                        p = row
                        q = column

            diagonal_scale = max(
                1.0,
                max(
                    abs(float(a[i, i]))
                    for i in range(n)
                ),
            )

            if (
                largest_value
                <= tolerance * diagonal_scale
            ):
                eigenvalues = np.array(
                    [
                        float(a[i, i])
                        for i in range(n)
                    ],
                    dtype=np.float64,
                )

                return (
                    eigenvalues,
                    eigenvectors,
                    iteration,
                    largest_value,
                )

            app = float(a[p, p])
            aqq = float(a[q, q])
            apq = float(a[p, q])

            if abs(apq) <= np.finfo(
                np.float64
            ).eps:
                a[p, q] = 0.0
                a[q, p] = 0.0
                continue

            tau = (
                aqq - app
            ) / (
                2.0 * apq
            )

            if tau >= 0.0:
                rotation_tangent = 1.0 / (
                    tau
                    + np.sqrt(
                        1.0 + tau * tau
                    )
                )
            else:
                rotation_tangent = -1.0 / (
                    -tau
                    + np.sqrt(
                        1.0 + tau * tau
                    )
                )

            cosine = 1.0 / np.sqrt(
                1.0
                + rotation_tangent
                * rotation_tangent
            )

            sine = (
                rotation_tangent
                * cosine
            )

            # Rotate the p and q rows/columns.
            for k in range(n):
                if k == p or k == q:
                    continue

                akp = float(a[k, p])
                akq = float(a[k, q])

                new_kp = (
                    cosine * akp
                    - sine * akq
                )

                new_kq = (
                    sine * akp
                    + cosine * akq
                )

                a[k, p] = new_kp
                a[p, k] = new_kp

                a[k, q] = new_kq
                a[q, k] = new_kq

            a[p, p] = (
                cosine * cosine * app
                - 2.0
                * sine
                * cosine
                * apq
                + sine * sine * aqq
            )

            a[q, q] = (
                sine * sine * app
                + 2.0
                * sine
                * cosine
                * apq
                + cosine * cosine * aqq
            )

            a[p, q] = 0.0
            a[q, p] = 0.0

            # Accumulate the eigenvector rotations.
            vector_p = (
                eigenvectors[:, p]
                .copy()
            )

            vector_q = (
                eigenvectors[:, q]
                .copy()
            )

            eigenvectors[:, p] = (
                cosine * vector_p
                - sine * vector_q
            )

            eigenvectors[:, q] = (
                sine * vector_p
                + cosine * vector_q
            )

        raise RuntimeError(
            "Jacobi eigen-decomposition did not converge "
            f"after {max_iterations} iterations"
        )


    (
        eigenvalues,
        eigenvectors,
        jacobi_iterations,
        final_off_diagonal,
    ) = jacobi_eigh_symmetric(
        covariance
    )

    order = np.argsort(
        eigenvalues
    )[::-1]

    eigenvalues = eigenvalues[
        order
    ]

    eigenvectors = eigenvectors[
        :,
        order,
    ]

    # Numerical round-off can produce tiny negative eigenvalues.
    cleaned_eigenvalues = np.where(
        eigenvalues > 0.0,
        eigenvalues,
        0.0,
    )

    components = eigenvectors[
        :,
        :2,
    ]

    # Project onto the first two components without BLAS-backed @.
    projection = np.empty(
        (number_of_rows, 2),
        dtype=np.float64,
    )
    
    for component_index in range(2):
        coordinate = np.zeros(
            number_of_rows,
            dtype=np.float64,
        )
    
        for feature_index in range(number_of_features):
            coordinate += (
                x_centered[:, feature_index]
                * components[
                    feature_index,
                    component_index,
                ]
            )
    
        projection[:, component_index] = coordinate
    
    print(
        "Projection calculation passed",
        flush=True,
    )

    if not np.isfinite(projection).all():
        raise ValueError(
            "PCA projection contains non-finite values"
        )

    total_variance = float(
        cleaned_eigenvalues.sum()
    )

    if total_variance > 0.0:
        explained_ratio = (
            cleaned_eigenvalues[:2]
            / total_variance
        )
    else:
        explained_ratio = np.array(
            [np.nan, np.nan],
            dtype=np.float64,
        )

    print(
        "Jacobi decomposition converged in "
        f"{jacobi_iterations} iterations; "
        f"final maximum off-diagonal value="
        f"{final_off_diagonal:.3e}",
        flush=True,
    )

    print(
        "6. PCA calculation passed",
        flush=True,
    )

    output = desc[
        ["molecule_id"]
    ].copy()

    output["PC1"] = projection[
        :,
        0,
    ]

    output["PC2"] = projection[
        :,
        1,
    ]

    output_path = (
        root
        / "data"
        / "processed"
        / "descriptor_pca.csv"
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        output_path.with_suffix(
            ".csv.tmp"
        )
    )

    output.to_csv(
        temporary_path,
        index=False,
    )

    temporary_path.replace(
        output_path
    )

    metadata = {
        "method": "robust-scaled covariance PCA",
        "eigendecomposition": (
            "Jacobi symmetric rotation algorithm without numpy.linalg"
        ),
        "seed": seed,
        "number_of_molecules": int(
            len(output)
        ),
        "descriptor_columns": (
            numeric.columns.tolist()
        ),
        "descriptor_count": int(
            numeric.shape[1]
        ),
        "covariance_shape": list(
            covariance.shape
        ),
        "jacobi_iterations": int(
            jacobi_iterations
        ),
        "jacobi_final_maximum_off_diagonal": float(
            final_off_diagonal
        ),
        "eigenvalues": (
            eigenvalues.tolist()
        ),
        "explained_variance_ratio": (
            explained_ratio.tolist()
        ),
    }

    metadata_path = (
        root
        / "data"
        / "metadata"
        / "descriptor_pca_metadata.json"
    )

    metadata_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    metadata_path.write_text(
        json.dumps(
            metadata,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        f"7. Created: {output_path}",
        flush=True,
    )

    print(
        f"8. Created: {metadata_path}",
        flush=True,
    )

    print(
        "Explained variance ratio:",
        explained_ratio,
        flush=True,
    )
    '''
)

# This file is regenerated whenever this notebook cell runs.
worker_path.write_text(
    worker_code,
    encoding="utf-8",
)

worker_environment = os.environ.copy()

worker_environment[
    "TOX_PROJECT_ROOT"
] = str(ROOT)

worker_environment[
    "TOX_PROJECT_SEED"
] = str(SEED)

worker_environment[
    "OMP_NUM_THREADS"
] = "1"

worker_environment[
    "MKL_NUM_THREADS"
] = "1"

worker_environment[
    "OPENBLAS_NUM_THREADS"
] = "1"

worker_environment[
    "NUMEXPR_NUM_THREADS"
] = "1"

result = subprocess.run(
    [
        sys.executable,
        "-X",
        "faulthandler",
        str(worker_path),
    ],
    cwd=str(ROOT),
    env=worker_environment,
    capture_output=True,
    text=True,
)

print(
    "Worker Python:",
    sys.executable,
)

print(
    "Worker exit code:",
    result.returncode,
)

print("\nSTDOUT:")
print(
    result.stdout
    if result.stdout
    else "(no standard output)"
)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "The isolated PCA worker failed. "
        f"Exit code: {result.returncode}. "
        "Review the worker STDOUT and STDERR above. "
        "The Jupyter kernel remained protected."
    )

pca_path = (
    ROOT
    / "data"
    / "processed"
    / "descriptor_pca.csv"
)

metadata_path = (
    ROOT
    / "data"
    / "metadata"
    / "descriptor_pca_metadata.json"
)

for artifact in [
    pca_path,
    metadata_path,
]:
    if not artifact.exists():
        raise FileNotFoundError(
            f"Expected artifact was not created: {artifact}"
        )

    if artifact.stat().st_size == 0:
        raise RuntimeError(
            f"Created artifact is empty: {artifact}"
        )

proj = pd.read_csv(
    pca_path,
    dtype={"molecule_id": "string"},
)

if list(proj.columns) != [
    "molecule_id",
    "PC1",
    "PC2",
]:
    raise ValueError(
        f"Unexpected PCA columns: {proj.columns.tolist()}"
    )

if len(proj) != 25583:
    print(
        "Warning: PCA row count differs from the previous "
        f"25,583-molecule run: {len(proj):,}"
    )

if proj[
    ["PC1", "PC2"]
].isna().any().any():
    raise ValueError(
        "PCA output contains missing coordinate values"
    )

print(
    "PCA output shape:",
    proj.shape,
)

print(
    f"Created: {pca_path.relative_to(ROOT)} "
    f"({pca_path.stat().st_size:,} bytes)"
)

print(
    f"Created: {metadata_path.relative_to(ROOT)} "
    f"({metadata_path.stat().st_size:,} bytes)"
)

display(
    proj.head()
)

Worker Python: D:\Users\anaconda3\envs\toxicity-screening\python.exe
Worker exit code: 0

STDOUT:
1. Reading endpoint records
2. Unique molecules: 25,583
3. Descriptor alignment passed
4. Descriptor matrix: (25583, 16)
5. Robust scaling passed
Covariance calculation passed
Projection calculation passed
Jacobi decomposition converged in 398 iterations; final maximum off-diagonal value=3.224e-10
6. PCA calculation passed
7. Created: D:\Dropbox\Work\Learning\Python\toxicity_screening_project\data\processed\descriptor_pca.csv
8. Created: D:\Dropbox\Work\Learning\Python\toxicity_screening_project\data\metadata\descriptor_pca_metadata.json
Explained variance ratio: [0.50558887 0.20955595]

PCA output shape: (25583, 3)
Created: data\processed\descriptor_pca.csv (1,735,125 bytes)
Created: data\metadata\descriptor_pca_metadata.json (1,213 bytes)


,molecule_id,PC1,PC2
0,PMWZSLRPVRCNKM-UHFFFAOYSA-N,-0.407799,0.083195
1,IOVNPTLMMFSPCO-UHFFFAOYSA-N,1.677925,1.641587
2,BSPCGOZVENCINN-AKAXFMLLSA-N,2.642549,-1.607110
3,UQBOMAVTFFNGLT-PVARCSIZSA-N,2.699387,-1.227625
4,DEYZGKXTVSBRIY-UHFFFAOYSA-N,1.678676,-0.043147


In [3]:
import os
import subprocess
import sys
import textwrap
from pathlib import Path

# Current notebook kernel:
# D:\Users\anaconda3\envs\toxicity-screening\python.exe
kernel_python = Path(sys.executable).resolve()

# Base Anaconda interpreter:
# D:\Users\anaconda3\python.exe
base_python = kernel_python.parents[2] / "python.exe"

if not base_python.exists():
    raise FileNotFoundError(
        f"Base Anaconda Python was not found: {base_python}"
    )

pca_path = (
    ROOT
    / "data"
    / "processed"
    / "descriptor_pca.csv"
)

if not pca_path.exists():
    raise FileNotFoundError(
        f"PCA data file was not found: {pca_path}"
    )

mpl_config_path = (
    ROOT
    / "reports"
    / "_matplotlib_config"
)

mpl_config_path.mkdir(
    parents=True,
    exist_ok=True,
)

plot_script = textwrap.dedent(
    r'''
    from pathlib import Path
    import csv
    import math
    import os

    os.environ["MPLBACKEND"] = "Agg"
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["OPENBLAS_NUM_THREADS"] = "1"
    os.environ["NUMEXPR_NUM_THREADS"] = "1"

    root = Path.cwd()

    input_path = (
        root
        / "data"
        / "processed"
        / "descriptor_pca.csv"
    )

    output_path = (
        root
        / "figures"
        / "descriptor_pca.png"
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "1. Reading PCA coordinates",
        flush=True,
    )

    pc1 = []
    pc2 = []

    with input_path.open(
        mode="r",
        encoding="utf-8",
        newline="",
    ) as handle:
        reader = csv.DictReader(handle)

        required_columns = {
            "molecule_id",
            "PC1",
            "PC2",
        }

        available_columns = set(
            reader.fieldnames or []
        )

        missing_columns = (
            required_columns
            - available_columns
        )

        if missing_columns:
            raise ValueError(
                "PCA CSV is missing columns: "
                f"{sorted(missing_columns)}"
            )

        for row in reader:
            x = float(row["PC1"])
            y = float(row["PC2"])

            if (
                math.isfinite(x)
                and math.isfinite(y)
            ):
                pc1.append(x)
                pc2.append(y)

    if not pc1:
        raise ValueError(
            "No finite PCA coordinates were found"
        )

    print(
        f"2. Loaded {len(pc1):,} coordinates",
        flush=True,
    )

    print(
        "3. Importing Matplotlib",
        flush=True,
    )

    import matplotlib

    matplotlib.use(
        "Agg",
        force=True,
    )

    import matplotlib.pyplot as plt

    print(
        "4. Matplotlib imported",
        flush=True,
    )

    fig, ax = plt.subplots(
        figsize=(7.0, 5.8),
    )

    ax.scatter(
        pc1,
        pc2,
        s=5,
        alpha=0.30,
        linewidths=0,
        rasterized=True,
    )

    ax.set_title(
        "Descriptor-Space PCA"
    )

    ax.set_xlabel(
        "PC1 (50.56% explained variance)"
    )

    ax.set_ylabel(
        "PC2 (20.96% explained variance)"
    )

    ax.grid(
        visible=True,
        alpha=0.20,
        linewidth=0.6,
    )

    fig.subplots_adjust(
        left=0.14,
        right=0.97,
        bottom=0.14,
        top=0.91,
    )

    print(
        "5. Saving PCA figure",
        flush=True,
    )

    fig.savefig(
        output_path,
        dpi=180,
        facecolor="white",
        bbox_inches="tight",
    )

    plt.close(fig)

    if not output_path.exists():
        raise FileNotFoundError(
            f"Figure was not created: {output_path}"
        )

    if output_path.stat().st_size == 0:
        raise RuntimeError(
            f"Created figure is empty: {output_path}"
        )

    print(
        f"6. Created: {output_path} "
        f"({output_path.stat().st_size:,} bytes)",
        flush=True,
    )
    '''
)

plot_environment = os.environ.copy()

plot_environment["MPLBACKEND"] = "Agg"
plot_environment["MPLCONFIGDIR"] = str(
    mpl_config_path
)

plot_environment["OMP_NUM_THREADS"] = "1"
plot_environment["MKL_NUM_THREADS"] = "1"
plot_environment["OPENBLAS_NUM_THREADS"] = "1"
plot_environment["NUMEXPR_NUM_THREADS"] = "1"

# Prevent unrelated packages from the user site directory
# from being injected into the base interpreter.
plot_environment["PYTHONNOUSERSITE"] = "1"

result = subprocess.run(
    [
        str(base_python),
        "-X",
        "faulthandler",
        "-c",
        plot_script,
    ],
    cwd=str(ROOT),
    env=plot_environment,
    capture_output=True,
    text=True,
)

print(
    "Notebook kernel Python:",
    kernel_python,
)

print(
    "Plotting Python:",
    base_python,
)

print(
    "Plot subprocess exit code:",
    result.returncode,
)

print("\nSTDOUT:")
print(
    result.stdout
    if result.stdout
    else "(no standard output)"
)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "The PCA plotting subprocess failed. "
        f"Exit code: {result.returncode}. "
        "Review the staged output above."
    )

figure_path = (
    ROOT
    / "figures"
    / "descriptor_pca.png"
)

if not figure_path.exists():
    raise FileNotFoundError(
        f"Expected figure was not created: {figure_path}"
    )

if figure_path.stat().st_size == 0:
    raise RuntimeError(
        f"Created figure is empty: {figure_path}"
    )

print(
    "PCA figure subprocess completed successfully."
)

print(
    f"Created: {figure_path.relative_to(ROOT)} "
    f"({figure_path.stat().st_size:,} bytes)"
)

Notebook kernel Python: D:\Users\anaconda3\envs\toxicity-screening\python.exe
Plotting Python: D:\Users\anaconda3\python.exe
Plot subprocess exit code: 0

STDOUT:
1. Reading PCA coordinates
2. Loaded 25,583 coordinates
3. Importing Matplotlib
4. Matplotlib imported
5. Saving PCA figure
6. Created: D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\descriptor_pca.png (144,374 bytes)

PCA figure subprocess completed successfully.
Created: figures\descriptor_pca.png (144,374 bytes)


In [4]:
from itertools import combinations

import numpy as np
import pandas as pd

from rdkit import DataStructs
from toxicity_screening.fingerprints import morgan_bit_vector


# ------------------------------------------------------------------
# 1. Reload the data into the notebook kernel
# ------------------------------------------------------------------

records_path = (
    ROOT
    / "data"
    / "processed"
    / "endpoint_records.parquet"
)

if not records_path.exists():
    raise FileNotFoundError(
        f"Endpoint records were not found: {records_path}"
    )

records = pd.read_parquet(
    records_path
)

required_columns = {
    "molecule_id",
    "standardized_smiles",
    "scaffold",
    "endpoint",
    "label",
}

missing_columns = (
    required_columns
    - set(records.columns)
)

if missing_columns:
    raise ValueError(
        "endpoint_records.parquet is missing required columns: "
        f"{sorted(missing_columns)}"
    )

records = records.copy()

records["molecule_id"] = (
    records["molecule_id"]
    .astype("string")
)

records["standardized_smiles"] = (
    records["standardized_smiles"]
    .astype("string")
)

records["endpoint"] = (
    records["endpoint"]
    .astype("string")
)

records["label"] = pd.to_numeric(
    records["label"],
    errors="coerce",
)

molecules = (
    records[
        [
            "molecule_id",
            "standardized_smiles",
            "scaffold",
        ]
    ]
    .drop_duplicates("molecule_id")
    .reset_index(drop=True)
)

print(
    f"Loaded records: {len(records):,}"
)

print(
    f"Unique molecules: {len(molecules):,}"
)


# ------------------------------------------------------------------
# 2. Prepare output directories
# ------------------------------------------------------------------

tables_directory = (
    ROOT
    / "tables"
)

metadata_directory = (
    ROOT
    / "data"
    / "metadata"
)

tables_directory.mkdir(
    parents=True,
    exist_ok=True,
)

metadata_directory.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------------
# 3. Scaffold-frequency analysis
# ------------------------------------------------------------------

scaffolds = (
    records
    .groupby(
        [
            "endpoint",
            "scaffold",
        ],
        dropna=False,
    )
    .agg(
        records=(
            "molecule_id",
            "size",
        ),
        positives=(
            "label",
            lambda values: int(
                values.eq(1).sum()
            ),
        ),
        prevalence=(
            "label",
            "mean",
        ),
    )
    .reset_index()
)

scaffolds = (
    scaffolds
    .sort_values(
        [
            "endpoint",
            "records",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

scaffold_path = (
    tables_directory
    / "scaffold_frequency.csv"
)

scaffolds.to_csv(
    scaffold_path,
    index=False,
)


# ------------------------------------------------------------------
# 4. Endpoint-overlap matrix
#
# Do not use:
#     presence.T.dot(presence)
#
# The @/dot operation may invoke the defective BLAS DLL.
# ------------------------------------------------------------------

presence = (
    records[
        [
            "molecule_id",
            "endpoint",
        ]
    ]
    .dropna()
    .drop_duplicates()
    .assign(present=np.uint8(1))
    .pivot(
        index="molecule_id",
        columns="endpoint",
        values="present",
    )
    .fillna(0)
    .astype(np.uint8)
)

endpoint_names = (
    presence.columns
    .astype(str)
    .tolist()
)

overlap = pd.DataFrame(
    0,
    index=endpoint_names,
    columns=endpoint_names,
    dtype=np.int64,
)

presence_arrays = {
    endpoint: presence[endpoint].to_numpy(
        dtype=np.uint8,
        copy=False,
    )
    for endpoint in endpoint_names
}

for first_index, first_endpoint in enumerate(
    endpoint_names
):
    first_values = presence_arrays[
        first_endpoint
    ]

    for second_index in range(
        first_index,
        len(endpoint_names),
    ):
        second_endpoint = endpoint_names[
            second_index
        ]

        second_values = presence_arrays[
            second_endpoint
        ]

        shared_count = int(
            np.count_nonzero(
                (first_values == 1)
                & (second_values == 1)
            )
        )

        overlap.loc[
            first_endpoint,
            second_endpoint,
        ] = shared_count

        overlap.loc[
            second_endpoint,
            first_endpoint,
        ] = shared_count

overlap.index.name = "endpoint"

overlap_path = (
    tables_directory
    / "endpoint_overlap.csv"
)

overlap.to_csv(
    overlap_path
)


# ------------------------------------------------------------------
# 5. Molecular-fingerprint similarity sample
# ------------------------------------------------------------------

valid_molecules = (
    molecules
    .dropna(
        subset=[
            "molecule_id",
            "standardized_smiles",
        ]
    )
    .loc[
        lambda frame:
        frame["standardized_smiles"].str.len() > 0
    ]
    .reset_index(drop=True)
)

sample_limit = (
    1000
    if PROFILE == "smoke"
    else 3000
)

sample_size = min(
    len(valid_molecules),
    sample_limit,
)

if sample_size < 2:
    raise ValueError(
        "At least two valid molecules are required "
        "for similarity analysis"
    )

sample = (
    valid_molecules
    .sample(
        n=sample_size,
        random_state=SEED,
    )
    .reset_index(drop=True)
)

print(
    f"Generating fingerprints for "
    f"{len(sample):,} molecules..."
)

fps = [
    morgan_bit_vector(
        str(smiles)
    )
    for smiles in sample[
        "standardized_smiles"
    ]
]

if any(
    fingerprint is None
    for fingerprint in fps
):
    failed_fingerprints = sum(
        fingerprint is None
        for fingerprint in fps
    )

    raise ValueError(
        f"Fingerprint generation failed for "
        f"{failed_fingerprints} molecules"
    )

maximum_pairs = (
    len(sample)
    * (len(sample) - 1)
    // 2
)

number_of_pairs = min(
    5000,
    maximum_pairs,
)

rng = np.random.default_rng(
    SEED
)

if number_of_pairs == maximum_pairs:
    pair_indices = list(
        combinations(
            range(len(sample)),
            2,
        )
    )
else:
    selected_pairs = set()

    while len(selected_pairs) < number_of_pairs:
        i = int(
            rng.integers(
                0,
                len(sample),
            )
        )

        j = int(
            rng.integers(
                0,
                len(sample),
            )
        )

        if i == j:
            continue

        if i > j:
            i, j = j, i

        selected_pairs.add(
            (i, j)
        )

    pair_indices = sorted(
        selected_pairs
    )

rows = []

for i, j in pair_indices:
    rows.append(
        {
            "molecule_a": sample.at[
                i,
                "molecule_id",
            ],
            "molecule_b": sample.at[
                j,
                "molecule_id",
            ],
            "tanimoto": float(
                DataStructs.TanimotoSimilarity(
                    fps[i],
                    fps[j],
                )
            ),
        }
    )

similarity_sample = pd.DataFrame(
    rows
)

similarity_path = (
    metadata_directory
    / "similarity_sample.csv"
)

similarity_sample.to_csv(
    similarity_path,
    index=False,
)


# ------------------------------------------------------------------
# 6. Validate and report artifacts
# ------------------------------------------------------------------

for artifact in [
    scaffold_path,
    overlap_path,
    similarity_path,
]:
    if not artifact.exists():
        raise FileNotFoundError(
            f"Expected artifact was not created: {artifact}"
        )

    if artifact.stat().st_size == 0:
        raise RuntimeError(
            f"Created artifact is empty: {artifact}"
        )

print()
print(
    f"Scaffold rows: {len(scaffolds):,}"
)

print(
    f"Overlap matrix shape: {overlap.shape}"
)

print(
    f"Similarity pairs: {len(similarity_sample):,}"
)

print(
    f"Created: {scaffold_path.relative_to(ROOT)}"
)

print(
    f"Created: {overlap_path.relative_to(ROOT)}"
)

print(
    f"Created: {similarity_path.relative_to(ROOT)}"
)

display(overlap)
display(scaffolds.head(10))
display(similarity_sample.head())

Loaded records: 50,612
Unique molecules: 25,583
Generating fingerprints for 1,000 molecules...

Scaffold rows: 16,482
Overlap matrix shape: (6, 6)
Similarity pairs: 5,000
Created: tables\scaffold_frequency.csv
Created: tables\endpoint_overlap.csv
Created: data\metadata\similarity_sample.csv


,SR-ARE,SR-ATAD5,SR-MMP,SR-p53,ames_mutagenicity,herg_blockade
endpoint,,,,,,
SR-ARE,7596,7594,7590,7592,1889,318
SR-ATAD5,7594,7610,7604,7607,1893,318
SR-MMP,7590,7604,7604,7602,1892,318
SR-p53,7592,7607,7602,7607,1893,318
ames_mutagenicity,1889,1893,1892,1893,7246,76
herg_blockade,318,318,318,318,76,12949


,endpoint,scaffold,records,positives,prevalence
0,SR-ARE,,1713,135,0.091899
1,SR-ARE,c1ccccc1,1456,141,0.116915
2,SR-ARE,c1ccncc1,88,7,0.094595
3,SR-ARE,c1ccc(Cc2ccccc2)cc1,81,15,0.288462
4,SR-ARE,c1ccc2ccccc2c1,66,11,0.207547
5,SR-ARE,C1CCCCC1,63,2,0.036364
6,SR-ARE,c1ccc(Oc2ccccc2)cc1,44,8,0.216216
7,SR-ARE,O=C1C=C2CCC3C4CCCC4CCC3C2CC1,43,8,0.347826
8,SR-ARE,O=C1C=CC2C(=C1)CCC1C3CCCC3CCC21,41,3,0.120000
9,SR-ARE,C1=CCCCC1,38,3,0.096774


,molecule_a,molecule_b,tanimoto
0,FNRMQXKIOYLKTO-VWLOTQADSA-N,WJRBRSLFGCUECM-UHFFFAOYSA-N,0.039474
1,FNRMQXKIOYLKTO-VWLOTQADSA-N,LUGNTHCVMDLTQG-UHFFFAOYSA-N,0.111111
2,FNRMQXKIOYLKTO-VWLOTQADSA-N,ZLAONGKKHFSBFL-UHFFFAOYSA-N,0.083333
3,FNRMQXKIOYLKTO-VWLOTQADSA-N,DQDFRZYUWQHTEY-UHFFFAOYSA-N,0.219780
4,FNRMQXKIOYLKTO-VWLOTQADSA-N,KQXKVJAGOJTNJS-UHFFFAOYSA-N,0.142857


### Completion gate
Confirm that the declared artifacts exist before continuing to `08_split_generation.ipynb`.